# 🥈 Couche Silver : Nettoyage, Typage & Enrichissement avec PySpark

---

## 🎯 Objectifs de ce Notebook
1. Identifier les anomalies du monde réel dans les données brutes (valeurs négatives, outliers, incohérences).
2. Appliquer les règles de nettoyage et de conformité métier.
3. Calculer des colonnes dérivées à forte valeur ajoutée (`trip_duration_minutes`, `speed_mph`, `tip_rate`).
4. Joindre les trajets avec le référentiel des zones géographiques (Boroughs de New York).
5. Sauvegarder les données nettoyées dans le bucket MinIO `silver`, optimisées en **Parquet partitionné par date**.


---
## 1. Initialisation de la Session PySpark


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

minio_endpoint = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.getenv("MINIO_ACCESS_KEY", "admin")
minio_secret_key = os.getenv("MINIO_SECRET_KEY", "password123")

spark = (
    SparkSession.builder
    .appName("02_Transformation_Silver")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark prêt pour la couche Silver !")


---
## 2. Chargement des Données Brutes depuis la Couche Bronze


In [ ]:
# Chemins S3A vers le bucket Bronze
bronze_taxi_path = "s3a://bronze/nyc_taxi/raw_year=2024/raw_month=01/*.parquet"
bronze_zone_path = "s3a://bronze/reference/taxi_zone_lookup.csv"

df_raw = spark.read.parquet(bronze_taxi_path)
df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv(bronze_zone_path)

total_raw_count = df_raw.count()
print(f"📥 Données brutes chargées : {total_raw_count:,} courses.")


---
## 3. Analyse Exploratoire de la Qualité des Données (Data Quality Audit)

Pourquoi la couche Silver est-elle indispensable ? Découvrons les anomalies présentes dans les données réelles de New York :


In [ ]:
# 1. Montants négatifs (remboursements, contestations, erreurs de saisie)
neg_fares = df_raw.filter(F.col("fare_amount") < 0).count()
neg_total = df_raw.filter(F.col("total_amount") < 0).count()

# 2. Nombre de passagers nul ou aberrant (> 6 pour un taxi standard)
zero_passengers = df_raw.filter((F.col("passenger_count") == 0) | F.col("passenger_count").isNull()).count()

# 3. Distance nulle alors que la course est facturée
zero_dist_high_fare = df_raw.filter((F.col("trip_distance") <= 0) & (F.col("fare_amount") > 10)).count()

# 4. Dates incohérentes (en dehors de Janvier 2024)
bad_dates = df_raw.filter(
    (F.col("tpep_pickup_datetime") < "2024-01-01") | 
    (F.col("tpep_pickup_datetime") >= "2024-02-01")
).count()

print("🔍 === AUDIT DE QUALITÉ DES DONNÉES BRUTES ===")
print(f"• Courses avec montant 'fare_amount' négatif    : {neg_fares:,}")
print(f"• Courses avec montant total négatif            : {neg_total:,}")
print(f"• Courses avec 0 ou aucun passager              : {zero_passengers:,}")
print(f"• Distance nulle mais montant > $10             : {zero_dist_high_fare:,}")
print(f"• Dates aberrantes (hors janvier 2024)          : {bad_dates:,}")


Affichons quelques exemples de courses aberrantes pour illustrer le problème aux élèves :


In [ ]:
print("Exemples de montants négatifs dans les données brutes :")
df_raw.filter(F.col("total_amount") < 0).select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"
).show(5)


---
## 4. Pipeline de Nettoyage et de Filtrage Métier

Nous appliquons les règles de filtrage définies avec le métier :
1. **Période valide** : Prises en charge strictement comprises entre le `2024-01-01 00:00:00` et le `2024-01-31 23:59:59`.
2. **Finances saines** : `fare_amount > 0` et `total_amount > 0`.
3. **Passagers réalistes** : `passenger_count >= 1` et `passenger_count <= 8`.
4. **Distance réaliste** : `trip_distance > 0` et `trip_distance <= 100` miles.
5. **Durée de course cohérente** : Entre 1 minute et 180 minutes (3 heures).


In [ ]:
# Calcul de la durée du trajet en minutes
df_with_duration = df_raw.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
)

# Application des filtres de qualité
df_cleaned = df_with_duration.filter(
    (F.col("tpep_pickup_datetime") >= "2024-01-01") &
    (F.col("tpep_pickup_datetime") < "2024-02-01") &
    (F.col("fare_amount") > 0) &
    (F.col("total_amount") > 0) &
    (F.col("passenger_count") >= 1) &
    (F.col("passenger_count") <= 8) &
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 100) &
    (F.col("trip_duration_minutes") >= 1.0) &
    (F.col("trip_duration_minutes") <= 180.0)
)

cleaned_count = df_cleaned.count()
rejected_count = total_raw_count - cleaned_count
pct_retained = (cleaned_count / total_raw_count) * 100

print(f"✅ Courses conservées dans la couche Silver : {cleaned_count:,} ({pct_retained:.1f}%)")
print(f"🗑️ Anomalies filtrées                     : {rejected_count:,} ({100 - pct_retained:.1f}%)")


---
## 5. Enrichissement des Données (Feature Engineering)

Ajoutons des colonnes calculées indispensables pour les analystes métiers :
- `pickup_date` : Date du trajet (servira également de clé de partitionnement).
- `pickup_hour` : Heure de prise en charge (0 à 23).
- `pickup_day_of_week` : Nom du jour de la semaine (Mon, Tue, ...).
- `speed_mph` : Vitesse moyenne du véhicule en miles par heure (`distance / (durée / 60)`).
- `tip_rate` : Ratio du pourboire par rapport au tarif de base (`tip_amount / fare_amount`).


In [ ]:
df_enriched = (
    df_cleaned
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) # 1=Dimanche, 2=Lundi...
    .withColumn("day_name", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("speed_mph", F.round(F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0), 2))
    .withColumn("tip_rate", F.round(F.col("tip_amount") / F.col("fare_amount"), 4))
)

# Remplacement des valeurs de vitesse aberrantes (ex: bouchon extrême ou GPS sauté)
df_enriched = df_enriched.filter(F.col("speed_mph") <= 85.0)

print("Aperçu des données enrichies :")
df_enriched.select(
    "pickup_date", "pickup_hour", "day_name", "trip_distance",
    "trip_duration_minutes", "speed_mph", "fare_amount", "tip_rate"
).show(5)


---
## 6. Jointure avec le Référentiel des Zones (Enrichissement Géographique)


In [ ]:
# Préparation de la table des zones
zones_pu = df_zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone")
)

zones_do = df_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone")
)

# Jointure pour la zone de départ et la zone d'arrivée
df_silver_final = (
    df_enriched
    .join(zones_pu, on="PULocationID", how="left")
    .join(zones_do, on="DOLocationID", how="left")
    .fillna({"pickup_borough": "Unknown", "dropoff_borough": "Unknown"})
)

print("Aperçu final avec arrondissements (Boroughs) :")
df_silver_final.select(
    "pickup_date", "pickup_borough", "dropoff_borough", "trip_distance", "total_amount"
).show(5)


---
## 7. Sauvegarde dans MinIO Silver (Parquet Partitionné)

### 💡 Pourquoi partitionner par date ?
En partitionnant par `pickup_date`, PySpark organise le stockage physique en sous-dossiers :
`s3a://silver/nyc_taxi/pickup_date=2024-01-01/part-*.parquet`
`s3a://silver/nyc_taxi/pickup_date=2024-01-02/part-*.parquet`

Lors d'une requête analytique sur une journée spécifique, le moteur ne lira **que** le sous-dossier correspondant (**Partition Pruning**), évitant de scanner plusieurs gigaoctets inutilement !


In [ ]:
silver_output_path = "s3a://silver/nyc_taxi"

print(f"💾 Écriture en cours vers {silver_output_path} (partitionné par pickup_date)...")

(
    df_silver_final
    .write
    .mode("overwrite")
    .partitionBy("pickup_date")
    .parquet(silver_output_path)
)

print("✅ Couche Silver écrite avec succès dans MinIO !")


Vérifions la présence des partitions écrites dans MinIO :


In [ ]:
import boto3

s3_client = boto3.client(
    "s3",
    endpoint_url=minio_endpoint,
    aws_access_key_id=minio_access_key,
    aws_secret_access_key=minio_secret_key,
    region_name="us-east-1"
)

response = s3_client.list_objects_v2(Bucket="silver", Prefix="nyc_taxi/pickup_date=", MaxKeys=10)
print("Exemple de partitions créées dans le bucket Silver :")
for obj in response.get("Contents", []):
    print(f"  • {obj['Key']} ({obj['Size'] / 1024:.1f} KB)")


---
## 🏁 Conclusion du Notebook 02
- Nous avons transformé des données brutes bruitées en un jeu de données **Silver** propre, standardisé et enrichi.
- Les données sont stockées au format **Parquet compressé avec Snappy**, partitionné par date pour des performances optimales.

👉 Passez au notebook suivant : **`03_chargement_gold.ipynb`** pour calculer les agrégats décisionnels et alimenter notre Data Warehouse PostgreSQL !
